In [1]:
import os
import sys
from bs4 import BeautifulSoup
from urllib.parse import urlparse


In [25]:


def is_external(url):
    # Check if URL starts with an external scheme.
    parsed = urlparse(url)
    return parsed.scheme in ('http', 'https', 'mailto')


def check_local_file(base_dir, url):
    """
    Take in a URL pointing to local resource in the current filesystem and determine whether the contents are stored locally.
    Valid = we have it
    Broken = we don't
    """
    # Remove query and fragment parts using urlparse.
    parsed = urlparse(url)
    local_path = parsed.path

    # If the local_path is empty (for example, a hash link), we can treat it as valid.
    if not local_path:
        return "valid"

    # Join with base directory of HTML file.
    abs_path = os.path.normpath(os.path.join(base_dir, local_path))
    if os.path.exists(abs_path):
        return "valid"
    else:
        return "broken"


def process_html_file(file_path):

    """
    Given an HTML file, find URLs in any src/href attributes and check their availability.
    """
    base_dir = os.path.dirname(file_path)

    out = []
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "html.parser")
    except Exception as e:
        sys.stderr.write(f"Error reading {file_path}: {e}\n")
        return

    # Loop over tags that might contain URLs in attributes "href" or "src"
    for tag in soup.find_all():
        for attr in ("href", "src"):
            url = tag.get(attr)
            if not url:
                continue  # no URL provided

            # Determine the link status.
            if is_external(url):
                status = "external"
            else:
                # For local URLs, check if the file exists.
                status = check_local_file(base_dir, url)

            # Write the result block to the output file.
            out += [[file_path, url, status]]
            # output_fh.write(f"{file_path}\t{url}\t{status}\n")

    return out
    
def process_doc_set(html_directory):
    out = []
    for root, dirs, files in os.walk(html_directory):
        for filename in files:
            if filename.lower().endswith((".html", ".htm")):
                file_path = os.path.join(root, filename)
                out += process_html_file(file_path)

    
    print(out[:20])
    return out

def write_links_to_file(links, out_file):
    out = "File\tLink\tStatus\n"

    for link_entry in links:
        out += "\t".join(link_entry) + "\n"
        
    open(out_file, "w").write(out)

def load_links_from_file(file):
    link_file_lines = open(file, "r").readlines()[1:]
    out = []

    for line in link_file_lines:
        out += [line.strip().split("\t")]

    return out

In [14]:
java_doc_links = process_doc_set("../../SourceDocs/JavaDocs")

[['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#SpecificationIntro', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#whatIs', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#architecture', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#writingAgents', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#deployingAgents', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#entryPoint', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#starting', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#startup', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#onload', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#onattach', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#onunload', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#tooloptions', 'valid'], ['../../SourceDocs/JavaDocs/html/specs/jvmti.html', '#environments', 'val

In [21]:
write_links_to_file(java_doc_links, "notebook_java_links.txt")

In [26]:
test_links = load_links_from_file("notebook_java_links.txt")

In [29]:
broken_java_links = [link_entry for link_entry in java_doc_links if link_entry[2] == "broken"]

In [40]:
print(len(broken_java_links))

20555


In [41]:
print(len(java_doc_links) - len(broken_java_links))

1126370


In [39]:
broken_counts = {}

for link in broken_java_links:
    file, url, status = link

    # Chop off fragment -- it's irrelevant here and creates duplicate entries
    url = url.split("#")[0]

    if url not in broken_counts:
        broken_counts[url] = {}
        broken_counts[url]["count"] = 1
        broken_counts[url]["files"] = [file]
    else:
        broken_counts[url]["count"] += 1
        broken_counts[url]["files"] += file
        
sorted_broken_counts = sorted(broken_counts.items(), key=lambda item: item[1]['count'], reverse=True)

for url, data in sorted_broken_counts:
    print(f"URL: {url}, Count: {data['count']}")

URL: ../../../../script-dir/jquery-3.7.1.min.js, Count: 3770
URL: ../../../../../legal/copyright.html, Count: 3770
URL: ../../../../../script-dir/jquery-3.7.1.min.js, Count: 3365
URL: ../../../../../../legal/copyright.html, Count: 3365
URL: ../../../script-dir/jquery-3.7.1.min.js, Count: 1556
URL: ../../../../legal/copyright.html, Count: 1556
URL: ../../../../../../script-dir/jquery-3.7.1.min.js, Count: 1289
URL: ../../../../../../../legal/copyright.html, Count: 1289
URL: ../../legal/copyright.html, Count: 88
URL: ../script-dir/jquery-3.7.1.min.js, Count: 87
URL: ../../../../specs/security/standard-names.html, Count: 82
URL: ../../../../../specs/security/standard-names.html, Count: 68
URL: ../../../../../../../script-dir/jquery-3.7.1.min.js, Count: 58
URL: ../../../../../../../../legal/copyright.html, Count: 58
URL: ../../../../../../specs/security/standard-names.html, Count: 16
URL: script-dir/jquery-3.7.1.min.js, Count: 11
URL: ./../legal/copyright.html, Count: 11
URL: ../../../../..